In [3]:
import pandas as pd

In [11]:
import warnings
warnings.filterwarnings("ignore")

In [12]:
from nselib import capital_market as cm

# Define dates as strings in DD-MM-YYYY format
start_date = '01-06-2021'
end_date = '05-01-2022'

# Use the correct function: price_volume_data
stk_data = cm.price_volume_data(symbol='TATACOFFEE', from_date=start_date, to_date=end_date)

print(stk_data.head())

       Symbol Series         Date  PrevClose  OpenPrice  HighPrice  LowPrice  \
0  TATACOFFEE     EQ  05-Jan-2022     217.80     218.50      218.7    214.30   
1  TATACOFFEE     EQ  04-Jan-2022     214.05     214.90      220.8    212.45   
2  TATACOFFEE     EQ  03-Jan-2022     213.15     213.95      216.9    212.30   
3  TATACOFFEE     EQ  31-Dec-2021     208.50     208.90      216.2    208.40   
4  TATACOFFEE     EQ  30-Dec-2021     211.35     211.45      211.5    207.90   

   LastPrice  ClosePrice  AveragePrice TotalTradedQuantity        Turnover₹  \
0     214.65      214.85        215.72           13,50,483  29,13,25,687.40   
1     217.40      217.80        217.41           34,77,443  75,60,37,979.15   
2     214.30      214.05        214.76           15,25,259  32,75,66,528.25   
3     213.70      213.15        213.37           30,45,699  64,98,54,073.20   
4     208.80      208.50        209.36            9,77,484  20,46,47,553.50   

  No.ofTrades  
0      12,079  
1      28,87

In [13]:
stk_data=stk_data[["OpenPrice","HighPrice","LowPrice","ClosePrice"]]

In [14]:
stk_data

,OpenPrice,HighPrice,LowPrice,ClosePrice
0,218.50,218.70,214.30,214.85
1,214.90,220.80,212.45,217.80
2,213.95,216.90,212.30,214.05
3,208.90,216.20,208.40,213.15
4,211.45,211.50,207.90,208.50
...,...,...,...,...
146,176.40,176.65,173.00,174.35
147,177.90,177.90,173.75,174.35
148,176.90,178.70,175.60,176.70
149,173.55,175.65,172.05,174.00


In [15]:
column="ClosePrice"

In [16]:
from sklearn.preprocessing import MinMaxScaler
Ms = MinMaxScaler()
data1= Ms.fit_transform(stk_data)
print("Len:",data1.shape)

Len: (151, 4)


In [17]:
data1=pd.DataFrame(data1,columns=["Open","High","Low","Close"])

In [18]:
training_size = round(len(data1 ) * 0.80)
print(training_size)
X_train=data1[:training_size]
X_test=data1[training_size:]
print("X_train length:",X_train.shape)
print("X_test length:",X_test.shape)
y_train=data1[:training_size]
y_test=data1[training_size:]
print("y_train length:",y_train.shape)
print("y_test length:",y_test.shape)

121
X_train length: (121, 4)
X_test length: (30, 4)
y_train length: (121, 4)
y_test length: (30, 4)


In [19]:
performance_metrics={"Model":[],"RMSE":[],"MaPe":[],"Lag":[],"Test":[]}

In [27]:
#1.Function
def cominbation(dataset,listt):
    print(listt)
    datasetTwo=dataset[listt]
#2. train/test data split
    test_obs = 28
    train =datasetTwo[:-test_obs]
    test = datasetTwo[-test_obs:]
#3. manual Lag order scan - model fit
    from statsmodels.tsa.api import VAR
    for i in [1,2,3,4,5,6,7,8,9,10]:
        model = VAR(train)
        results = model.fit(i)
        print('Order =', i)
        print('AIC: ', results.aic)
        print('BIC: ', results.bic)
        print()
#4. Automatic order selection
    x = model.select_order(maxlags=12)
    order=x.selected_orders["aic"]
    result = model.fit(order)
#5. Forecasting
    lagged_Values = train.values[-order:]
    pred = result.forecast(y=lagged_Values,steps=28) 
#6. Save forecasting
    preds=pd.DataFrame(pred,columns=listt)
    preds.to_csv("varforecasted_{}.csv".format(test_obs))
#7. Scoring 
    from sklearn.metrics import mean_squared_error
    rmse= round(mean_squared_error(test,pred,squared=False))
    from sklearn.metrics import mean_absolute_percentage_error
    mape=mean_absolute_percentage_error(test,pred)
#8. Display results
    performance_metrics["Model"].append(listt)
    performance_metrics["RMSE"].append(rmse)
    performance_metrics["MaPe"].append(mape)
    performance_metrics["Lag"].append(order)
    performance_metrics["Test"].append(test_obs)
    perf=pd.DataFrame(performance_metrics)
    return perf,result,pred

In [28]:
listt=["Close","High","Open","Low"]